# Importing data

Importing data means taking data in some form, and preparing it so that we can express that data as nodes and edges. On its own, this is not too challenging - it mostly means converting data formats. The harder part is harmonizing the data, so that the fields used across imported databases are consistent enough that we can link consumers and supplier.

Let's make this more concrete with an example. In the file `lci-carbon-fiber.xlsx` we have data from the publication [Ecological assessment of fuel cell electric vehicles with special focus on type IV carbon fiber hydrogen tank](https://www.sciencedirect.com/science/article/abs/pii/S0959652620333229). As this data is from Excel, it is tabular, and so on its surface looks different than the graph:

<img src="images/spreadsheet.png">

However, this difference is mostly cosmetic. Both the _document_ and _graph_ perspectives are showing the same information, but with a different emphasis and organizing structure. In the graph perspective, edges are independent objects with their own metadata, and their sources and targets are given as [pointers](https://en.wikipedia.org/wiki/Pointer_(computer_programming)) to the node objects. In the document perspective, edges are subsumed in the definition of the nodes, and because most input data formats don't have pointers, references to input or output flows are defined by the attributes of thoses flows.

Because we only have flow attributes, we need to define a way that we associate those attributes with nodes in our existing databases. This is trickier than you might think, as those is no guarantee that two data providers will use the same labels for things like locations or units; indeed, sometimes we even find different labels for the same attributes.

Therefore, Brightway treats IO as a classic [ETL pipeline](https://en.wikipedia.org/wiki/Extract,_transform,_load), and applies a series of transformation functions to prepare the data and find the correct flows. Let's look at our real-world example:

In [ ]:
import bw2data as bd
import bw2io as bi
from pathlib import Path

The example data is built on top of ecoinvent. You should update the project name to a project with ecoinvent 3.10.1 already installed.

In [ ]:
bd.projects.set_current("cf-excel")

In [ ]:
imp = bi.ExcelImporter(Path.cwd() / "lci-carbon-fiber.xlsx")

Before we make any changes, let's see what the data looks like in its raw form:

In [ ]:
imp.data[0]

This is actually aleady quite close to the final form. In this case we are lucky as the import data was designed to be used in Brightway. Normally we would need to apply transformation functions; lets see what those default transformation functions would be:

In [ ]:
imp.strategies

We can apply these default strategies:

In [ ]:
imp.apply_strategies()

We can look at the imported data statistics:

In [ ]:
imp.statistics()

We can iterate over the unlinked edges to get a sense for what we are missing:

In [ ]:
for edge, _ in zip(imp.unlinked, range(5)):
    print(edge)

OK, some unlinked exchanges are clearly from ecoinvent. Let's try to link those.

In [ ]:
imp.match_database("ecoinvent-3.10.1-cutoff", fields=('name', 'reference product', 'unit', 'location'))
imp.statistics()

Let's check the unlinked edges:

In [ ]:
for edge in imp.unlinked:
    print(edge)

Let's look at the missing Argon flow first. Let's check our database layout:

In [ ]:
bd.databases

That should be in the `ecoinvent-3.10.1-biosphere` database. Let's search for argon:

In [ ]:
[x for x in bd.Database("ecoinvent-3.10.1-biosphere") if "argon" in x["name"].lower()]

OK, so we have the following. In our imported data:

```python
{
    'name': 'Argon-40', 
    'unit': 'kilogram', 
    'categories': ('air',),
}
```

And in our `ecoinvent-3.10-biosphere` database:

```python
{
    'name': 'Argon', 
    'unit': 'kilogram', 
    'categories': ('air',),
}
```

We can patch this manually. One option would be to use the `Migration` definitions:

In [ ]:
migration = {
    "fields": ["name", "categories"],
    "data": [
        (
            ("Argon-40", ("air",)),
            {
                "name": "Argon",
            },
        )
    ],
}

In [ ]:
bi.Migration(name="ei3.9-3.10").write(data=migration, description="ei 3.9 to 3.10")

In [ ]:
imp.data = bi.strategies.migrate_exchanges(
    db=imp.data,
    migration="ei3.9-3.10"
)

However, it would be just as easy here to write a specific function:

In [ ]:
def fix_argon(data):
    for ds in data:
        for edge in ds['exchanges']:
            if "input" not in edge and edge["name"] == "Argon-40" and edge["categories"] == ("air",):
                edge["name"] = "Argon"
    return data

In [ ]:
imp.apply_strategy(fix_argon)

For other changes, we have some builtin migrations to take care of exactly these kinds of discrepancies. This is the [randonneur](https://github.com/brightway-lca/randonneur) tool, and its accompanying [randonneur_data](https://github.com/brightway-lca/randonneur_data) set of pre-computed migrations. Let look and see what is available:

In [ ]:
import randonneur_data as rd
registry = rd.Registry()
list(registry)

We can sample the data in each migration:

In [ ]:
registry.sample('ecoinvent-3.9.1-cutoff-ecoinvent-3.10-cutoff')

In this specific case, the name difference was a change from ecoinvent version 3.9 to 3.10. We can apply the migration `ecoinvent-3.9.1-cutoff-ecoinvent-3.10-cutoff`, but need to be careful, as our unit labels don't match exactly. That's OK, we don't need to match against the unit.

In [ ]:
imp.randonneur(
    label='ecoinvent-3.9.1-cutoff-ecoinvent-3.10-cutoff',
    fields=['name', 'location', 'reference product'],
)

In [ ]:
imp.match_database("ecoinvent-3.10.1-cutoff", fields=('name', 'reference product', 'location'))
imp.match_database("ecoinvent-3.10.1-biosphere", fields=('name', 'unit', 'categories'))
imp.statistics()

In [ ]:
assert imp.all_linked
imp.write_database()

# Going step by step

## Step 1: Extraction

`bw2io` has a number of extractors:

- csv
- excel
- ecospold 1
- ecospold 2
- exiobase
- JSON-LD
- SimaPro CSV (via [bw_simapro_csv](https://github.com/brightway-lca/bw_simapro_csv))

An extract reads a given file format, and turns LCI data into the generic LCI format.

Let's look at a weird dataset, and write our own extractor:

In [ ]:
import json
from pathlib import Path

In [ ]:
data = json.load(open("pan.json"))
data

The first thing we will need to do is fix the labels.

Our basic labels are:

- comment
- exchanges
- location
- name
- reference product
- unit

**Exercise**: How could we change the wrong labels to the right ones?

In [ ]:
def extract_json_data(filepath: Path) -> list[dict]:
    data = json.load(open(filepath))
    ...
    return data

The next thing we need to do is fix the formatting of the edges. Right now they look like this:

```python
[
    'polyacrylonitrile production (PAN) by polymerisation',
    1,
    'RER',
    'kg'
]
```

This needs to be a dictionary. That dictionary should have the following keys:

- amount
- name (of linked process)
- location (of link process)
- reference product (of linked process)

**Exercise**: Extend the extraction function to process _each_ edge.

Finally, we need to add in some additional data which isn't present in the `pan.json` file. Specifically, we need to add the `type` of the node and its edges.

- The node should have the type `bd.labels.chimaera_node_default`
- The edge which the same name and location as its node should have the type `bd.labels.production_edge_default`
- The other edges should have the type `bd.labels.consumption_edge_default`

We can now build of new importer. We will build on top of the `bw2io` class:

In [ ]:
FIELD_MAPPING = {
    "a": "comment",
    "b": "location",
    "c": "reference product",
    "d": "unit",
    "e": "name",
    "f": "exchanges",
}

def edge_as_dict(obj: list) -> dict:
    return {
        "name": obj[0],
        "amount": obj[1],
        "location": obj[2],
        "unit": obj[3]
    }
    

def extract_json_data(filepath: Path) -> list[dict]:
    data = json.load(open(filepath))
    for ds in data:
        for key in list(ds):
            if key in FIELD_MAPPING:
                ds[FIELD_MAPPING[key]] = ds.pop(key)

        ds['type'] = bd.labels.chimaera_node_default

        edges = []
        for lst in ds["exchanges"]:
            edge = edge_as_dict(lst)
            if edge["name"] == ds["name"] and edge["location"] == ds["location"]:
                edge["type"] = bd.labels.production_edge_default
            else:
                edge["type"] = bd.labels.consumption_edge_default
            edges.append(edge)

        ds["exchanges"] = edges

    return data

In [ ]:
class WeirdJSONImporter(bi.importers.base_lci.LCIImporter):
    def extract_json_file(self, filepath: Path) -> None:
        self.data = extract_json_data(filepath=filepath)

We can now use this importer and perform extraction.

In [ ]:
imp = WeirdJSONImporter("PAN")

In [ ]:
imp.extract_json_file("pan.json")

In [ ]:
imp.data

## Step 2: Transformation

The `LCIImporter` base class has some built-in transformations:

In [ ]:
imp.strategies

These won't do much for us - however, `bw2io` has a large number of transformation functions:

- add_activity_hash_code
- add_cpc_classification_from_single_reference_product
- add_database_name
- assign_only_product_as_production
- assign_single_product_as_activity
- change_electricity_unit_mj_to_kwh
- clean_integer_codes
- convert_activity_parameters_to_list
- convert_uncertainty_types_to_integers
- create_composite_code
- create_products_as_new_nodes
- csv_add_missing_exchanges_section
- csv_drop_unknown
- csv_numerize
- csv_restore_booleans
- csv_restore_tuples
- delete_exchanges_missing_activity
- delete_ghost_exchanges
- delete_integer_codes
- delete_none_synonyms
- drop_falsey_uncertainty_fields_but_keep_zeros
- drop_temporary_outdated_biosphere_flows
- drop_unlinked
- drop_unlinked_cfs
- drop_unspecified_subcategories
- ensure_categories_are_tuples
- es1_allocate_multioutput
- es2_assign_only_product_with_amount_as_reference_product
- fix_ecoinvent_flows_pre35
- fix_localized_water_flows
- fix_unreasonably_high_lognormal_uncertainties
- fix_zero_allocation_products
- json_ld_add_activity_unit
- json_ld_add_products_as_activities
- json_ld_allocate_datasets
- json_ld_convert_unit_to_reference_unit
- json_ld_fix_process_type
- json_ld_get_activities_list_from_rawdata
- json_ld_get_normalized_exchange_locations
- json_ld_get_normalized_exchange_units
- json_ld_label_exchange_type
- json_ld_lcia_add_method_metadata
- json_ld_lcia_convert_to_list
- json_ld_lcia_reformat_cfs_as_exchanges
- json_ld_lcia_set_method_metadata
- json_ld_location_name
- json_ld_prepare_exchange_fields_for_linking
- json_ld_remove_fields
- json_ld_rename_metadata_fields
- link_biosphere_by_flow_uuid
- link_internal_technosphere_by_composite_code
- link_iterable_by_fields
- link_technosphere_based_on_name_unit_location
- link_technosphere_by_activity_hash
- match_against_only_available_in_given_context_tree
- match_against_top_level_context
- match_internal_simapro_simapro_with_unit_conversion
- match_subcategories
- normalize_biosphere_categories
- normalize_biosphere_names
- normalize_simapro_biosphere_categories
- normalize_simapro_biosphere_names
- normalize_simapro_labels_to_brightway_standard
- normalize_units
- override_process_name_using_single_functional_exchange
- remove_biosphere_location_prefix_if_flow_in_same_location
- remove_random_exchanges
- remove_uncertainty_from_negative_loss_exchanges
- remove_unnamed_parameters
- remove_useeio_products
- remove_zero_amount_coproducts
- remove_zero_amount_inputs_with_no_activity
- reparametrize_lognormal_to_agree_with_static_amount
- separate_processes_from_products
- set_biosphere_type
- set_code_by_activity_hash
- set_lognormal_loc_value
- set_metadata_using_single_functional_exchange
- sp_allocate_functional_products
- sp_allocate_products
- split_exchanges
- split_simapro_name_geo
- split_simapro_name_geo_curly_brackets
- strip_biosphere_exc_locations
- tupleize_categories
- update_ecoinvent_locations
- update_social_flows_in_older_consequential

Still, we can apply the default transformation strategies, and see the results:

In [ ]:
imp.apply_strategies()
imp.data

Transformation strategies can do anything - they only constraint is that they take an input argument `data`, which is a list of dictionaries, and that they return that a list of dictionaries when they are done with their processing.

**Exercise**: Write a transformation function which reduces the amount of steam needed by 5 percent (you are modelling a more efficient production site), add a note to the exchange that the amount was decreased (you chose the label for this note), and then apply that strategy function using `imp.apply_strategy(my_function)`.

The reason we are writing transformation strategy functions is to make links in the `exchanges`. Currently they only have attributes of the product or process they want to link to - we need to make the connection to those products or processes _explicit_. A _linked edge_ will have an `input` attribute, which refers to the `(database, code)` of the product or process.

We can check how many edges are unlinked:

In [ ]:
imp.statistics()

We can loop through the unlinked edges:

In [ ]:
for edge in imp.unlinked:
    print(edge)

We need to tell Brightway how to do linking - which attributes it should use to determine the _identity_ of an edge. The tricky bit here is that we don't have any guarantees of uniqueness for fields like name, location, or reference product. Instead, we usually need to use a combination of those attributes, and maybe some others. Let's see an example:

In [ ]:
from bw2io.strategies import link_iterable_by_fields
from copy import deepcopy

In [ ]:
unlinked_data = [
    {
        "name": "Steel Production",
        "database": "my_db",
        "code": "steel_001",
        "exchanges": [
            {
                "name": "Iron Ore",
                "location": "Australia",
                "amount": 1.5,
                "type": "technosphere",
                "unit": "kg"
            },
            {
                "name": "Coal",
                "location": "China",
                "amount": 0.8,
                "type": "technosphere",
                "unit": "kg"
            },
            {
                "name": "Carbon Dioxide",
                "categories": ("air",),
                "amount": 2.3,
                "type": "biosphere",
                "unit": "kg"
            }
        ]
    }
]

In [ ]:
target_database = [
    {
        "name": "Iron Ore",
        "location": "Australia",
        "database": "ecoinvent",
        "code": "iron_ore_au_001"
    },
    {
        "name": "Iron Ore",
        "location": "Brazil",
        "database": "ecoinvent",
        "code": "iron_ore_au_001"
    },
    {
        "name": "Carbon Dioxide",
        "categories": ("air",),
        "database": "biosphere3",
        "code": "co2_air"
    }
]

In [ ]:
linked_result = link_iterable_by_fields(
    deepcopy(unlinked_data),
    other=target_database,
    fields=["name", "location"]
)
linked_result

The function `link_iterable_by_fields` will protect you - it will only create a link if there is exactly one possible target for that link:

In [ ]:
link_iterable_by_fields(
    deepcopy(unlinked_data),
    other=target_database,
    fields=["name"]  # No `location`
)

### Additional input parameters for `link_iterable_by_fields`

1. `internal=True`

When to use:
- All datasets you want to link are in the same collection/list
- You want exchanges to link to other datasets within the same collection
- You don't have a separate "target database" - everything is together

How it works:
- Sets `other` to be the same as `unlinked` automatically
- Links exchanges in one dataset to other datasets in the same collection
- All datasets MUST have "database" and "code" fields

Example:
```python
data = [
   {"database": "db", "code": "A", "exchanges": [{"name": "B"}]},
   {"database": "db", "code": "B", "exchanges": []}
]
link_iterable_by_fields(data, internal=True, fields=["name"])
# Links exchange "B" in dataset A to dataset B
```

vs. external linking:
```python
unlinked = [{"exchanges": [{"name": "B"}]}]
other = [{"name": "B", "database": "db", "code": "B"}]
link_iterable_by_fields(unlinked, other=other, fields=["name"])
```

2. `edge_kinds`

When to use:
- You have multiple exchange types (technosphere, biosphere, production, etc.)
- You want to link only specific exchange types
- Different exchange types need different field combinations or target databases

How it works:
- Filters which exchanges to process based on their "type" field
- Can be a string (single type) or list (multiple types)
- Only exchanges with matching types will be linked

Example:
```python
# Link only technosphere exchanges
link_iterable_by_fields(
   data,
   other=technosphere_db,
   fields=["name", "location"],
   edge_kinds=["technosphere"]
)

# Link only biosphere exchanges (separate call)
link_iterable_by_fields(
   data,
   other=biosphere_db,
   fields=["name", "categories"],
   edge_kinds=["biosphere"]
)
```

Common exchange types:
- "technosphere": Intermediate products/services
- "biosphere": Elementary flows (emissions, resources)
- "production": Production exchanges
- "substitution": Substitution exchanges

3. `this_node_kinds` and `other_node_kinds`

When to use:
- Your database has different node types (process, product, elementary, etc.)
- You want to control which node types can link to which other node types
- You want to prevent certain linking patterns (e.g., process -> process)

How it works:
- this_node_kinds: Filters which datasets in `unlinked` to process
 (based on their "type" field)
- other_node_kinds: Filters which datasets in `other` can be linked to
 (based on their "type" field)
- Only exchanges in datasets matching this_node_kinds will be processed
- Only datasets matching other_node_kinds can be linked to

Example:
```python
# Link processes to products, but NOT processes to processes
link_iterable_by_fields(
   data,
   internal=True,
   fields=["name"],
   this_node_kinds=["process"],      # Only process nodes
   other_node_kinds=["product"]      # Can only link to product nodes
)
```

This prevents:
- Process A -> Process B (blocked by other_node_kinds)
- Product A -> Process B (blocked by this_node_kinds)

But allows:
- Process A -> Product B (matches both filters)

Common node types:
- "process": Activity/process nodes
- "product": Intermediate product nodes
- "elementary": Elementary flow nodes

### Combining Parameters

You can combine these parameters for complex scenarios:

```python
# Link technosphere exchanges from process nodes to product nodes internally
link_iterable_by_fields(
    data,
    internal=True,
    fields=["name"],
    this_node_kinds=["process"],
    other_node_kinds=["product"],
    edge_kinds=["technosphere"]
)
```

This will:
- Only process datasets with type="process" (this_node_kinds)
- Only link to datasets with type="product" (other_node_kinds)
- Only link exchanges with type="technosphere" (edge_kinds)
- Link within the same collection (internal=True)

**Exercise**: Using link_iterable_by_fields

In [ ]:
unlinked_data = [
    {
        "name": "Steel Production, Basic Oxygen Furnace",
        "location": "US",
        "database": "student_db",
        "code": "steel_bof_001",
        "exchanges": [
            {
                "name": "Iron Ore",
                "location": "Australia",
                "unit": "kg",
                "amount": 1.5,
                "type": "technosphere"
            },
            {
                "name": "Iron Ore",
                "location": "Brazil",
                "unit": "kg",
                "amount": 0.3,
                "type": "technosphere"
            },
            {
                "name": "Coal",
                "location": "US",
                "unit": "kg",
                "amount": 0.8,
                "type": "technosphere"
            },
            {
                "name": "Coal",
                "location": "China",
                "unit": "kg",
                "amount": 0.2,
                "type": "technosphere"
            },
            {
                "name": "Carbon Dioxide",
                "categories": ("air",),
                "unit": "kg",
                "amount": 2.3,
                "type": "biosphere"
            },
            {
                "name": "Carbon Dioxide",
                "categories": ("air", "unspecified"),
                "unit": "kg",
                "amount": 0.1,
                "type": "biosphere"
            },
            {
                "name": "Water",
                "location": "US",
                "unit": "L",
                "amount": 10.0,
                "type": "technosphere"
            },
            {
                "name": "Water",
                "location": "US",
                "unit": "m3",
                "amount": 0.01,
                "type": "technosphere"
            }
        ]
    },
    {
        "name": "Electricity Production, Natural Gas",
        "location": "Germany",
        "database": "student_db",
        "code": "elec_ng_001",
        "exchanges": [
            {
                "name": "Natural Gas",
                "location": "Russia",
                "unit": "MJ",
                "amount": 1.0,
                "type": "technosphere"
            },
            {
                "name": "Natural Gas",
                "location": "Norway",
                "unit": "MJ",
                "amount": 0.5,
                "type": "technosphere"
            },
            {
                "name": "Methane",
                "categories": ("air",),
                "unit": "kg",
                "amount": 0.05,
                "type": "biosphere"
            },
            {
                "name": "Methane",
                "categories": ("air", "unspecified"),
                "unit": "kg",
                "amount": 0.02,
                "type": "biosphere"
            }
        ]
    }
]

In [ ]:
target_database = [
    # Iron Ore - multiple locations
    {
        "name": "Iron Ore",
        "location": "Australia",
        "unit": "kg",
        "database": "ecoinvent",
        "code": "iron_ore_au_001"
    },
    {
        "name": "Iron Ore",
        "location": "Brazil",
        "unit": "kg",
        "database": "ecoinvent",
        "code": "iron_ore_br_001"
    },
    {
        "name": "Iron Ore",
        "location": "China",
        "unit": "kg",
        "database": "ecoinvent",
        "code": "iron_ore_cn_001"
    },
    # Coal - multiple locations
    {
        "name": "Coal",
        "location": "US",
        "unit": "kg",
        "database": "ecoinvent",
        "code": "coal_us_001"
    },
    {
        "name": "Coal",
        "location": "China",
        "unit": "kg",
        "database": "ecoinvent",
        "code": "coal_cn_001"
    },
    {
        "name": "Coal",
        "location": "Australia",
        "unit": "kg",
        "database": "ecoinvent",
        "code": "coal_au_001"
    },
    # Carbon Dioxide - different categories
    {
        "name": "Carbon Dioxide",
        "categories": ("air",),
        "unit": "kg",
        "database": "biosphere3",
        "code": "co2_air"
    },
    {
        "name": "Carbon Dioxide",
        "categories": ("air", "unspecified"),
        "unit": "kg",
        "database": "biosphere3",
        "code": "co2_air_unspec"
    },
    # Water - same location, different units
    {
        "name": "Water",
        "location": "US",
        "unit": "L",
        "database": "ecoinvent",
        "code": "water_us_L_001"
    },
    {
        "name": "Water",
        "location": "US",
        "unit": "m3",
        "database": "ecoinvent",
        "code": "water_us_m3_001"
    },
    {
        "name": "Water",
        "location": "US",
        "unit": "kg",
        "database": "ecoinvent",
        "code": "water_us_kg_001"
    },
    # Natural Gas - multiple locations
    {
        "name": "Natural Gas",
        "location": "Russia",
        "unit": "MJ",
        "database": "ecoinvent",
        "code": "ng_ru_001"
    },
    {
        "name": "Natural Gas",
        "location": "Norway",
        "unit": "MJ",
        "database": "ecoinvent",
        "code": "ng_no_001"
    },
    {
        "name": "Natural Gas",
        "location": "US",
        "unit": "MJ",
        "database": "ecoinvent",
        "code": "ng_us_001"
    },
    # Methane - different categories
    {
        "name": "Methane",
        "categories": ("air",),
        "unit": "kg",
        "database": "biosphere3",
        "code": "ch4_air"
    },
    {
        "name": "Methane",
        "categories": ("air", "unspecified"),
        "unit": "kg",
        "database": "biosphere3",
        "code": "ch4_air_unspec"
    }
]

**Exercise Part One**: Link all exchanges in `unlinked_data` to `target_database`
by determining the correct field combinations.

Questions to answer:

1. Which fields are needed to uniquely link Iron Ore exchanges?
2. Which fields are needed to uniquely link Coal exchanges?
3. Which fields are needed to uniquely link Carbon Dioxide exchanges?
4. Which fields are needed to uniquely link Water exchanges?
5. Which fields are needed to uniquely link Natural Gas exchanges?
6. Which fields are needed to uniquely link Methane exchanges?
7. Can you link all exchanges with a single field combination?
   If not, how would you handle different exchange types?

Hint: You will need to run `link_iterable_by_fields` more than once, splitting up `unlinked_data` by using the `edge_kinds` input parameter.

In [ ]:
result = link_iterable_by_fields(
    unlinked_data,
    other=target_database,
    fields=[<what goes here?>],
    edge_kinds=[<some label(s)>]
)

# If you need more than one shot
result = link_iterable_by_fields(
    result, # Take output from first function as input here
    other=target_database,
    fields=[<what goes here?>],
    edge_kinds=[<some label(s)>]
)

**Exercise Part Two**: Filtering by node type

Scenario: You have a database with different node types:
  - 'process' nodes: actual processes/activities
  - 'product' nodes: intermediate products
  - 'elementary' nodes: elementary flows

You want to link processes to products, but not processes to processes.

Understanding `this_node_kind`s and `other_node_kinds`:

  this_node_kinds:
    - Filters which DATASETS in 'unlinked' to process
    - Based on the dataset's 'type' field (not exchange type!)
    - Only exchanges in datasets matching this_node_kinds will be processed
    - Example: this_node_kinds=['process'] means only process datasets

  other_node_kinds:
    - Filters which DATASETS in 'other' can be linked TO
    - Based on the target dataset's 'type' field
    - Only datasets matching other_node_kinds can be linked to
    - Example: other_node_kinds=['product'] means can only link to products

  How they work together:
    - this_node_kinds=['process'] + other_node_kinds=['product']
      → Only process nodes can link, and only to product nodes
    - This prevents: process → process (blocked by other_node_kinds)
    - This allows: process → product (matches both filters)

When to use node type filtering:
  ✓ You have multiple node types and want to control linking patterns
  ✓ You want to prevent certain linking patterns (e.g., process→process)
  ✓ You want to enforce a specific data model structure
  ✓ You're working with complex databases (e.g., ecoinvent)

**Task:**

Desired behavior:
  - Link 'Steel Product' exchange -> product node ✓
  - Link 'Iron Ore Product' exchange -> product node ✓
  - Do NOT link 'Another Steel Process' exchange -> process node ✗Use this_node_kinds and other_node_kinds to control linking

In [ ]:
node_type_data = [
    {
        "name": "Steel Production Process",
        "type": "process",
        "database": "my_db",
        "code": "steel_proc_001",
        "exchanges": [
            {
                "name": "Steel Product",
                "amount": 1.0,
                "type": "technosphere"
            },
            {
                "name": "Iron Ore Product",
                "amount": 1.5,
                "type": "technosphere"
            },
            {
                "name": "Another Steel Process",
                "amount": 0.2,
                "type": "technosphere"
            }
        ]
    },
    {
        "name": "Steel Product",
        "type": "product",
        "database": "my_db",
        "code": "steel_prod_001",
        "exchanges": []
    },
    {
        "name": "Iron Ore Product",
        "type": "product",
        "database": "my_db",
        "code": "iron_ore_prod_001",
        "exchanges": []
    },
    {
        "name": "Another Steel Process",
        "type": "process",
        "database": "my_db",
        "code": "steel_proc_002",
        "exchanges": []
    },
    {
        "name": "Carbon Dioxide",
        "type": "elementary",
        "categories": ("air",),
        "database": "biosphere3",
        "code": "co2_air",
        "exchanges": []
    }
]

In [ ]:
link_iterable_by_fields(
    node_type_data,
    internal=True,  # Linking `node_type_data` against itself
    fields=[<you choose this>],
    this_node_kinds=[<you choose this>],
    other_node_kinds=[<you choose this>]
)

The `LCIImporter` has a shortcut to use the `link_iterable_by_fields` function:

`.match_database()`

This method takes the following arguments:

* db_name: Optional[str] = None
* fields: Optional[List[str]] = None
* ignore_categories: bool = False
* relink: bool = False
* kind: Union[str, List[str], NoneType] = None
* edge_kinds: Optional[List[str]] = None
* this_node_kinds: Optional[List[str]] = None
* other_node_kinds: Optional[List[str]] = None
* processes_to_products: bool = False

This is the basic loop when importing data in Brightway - write some transformation strategies to get data into the right form, then try to link, check what links are still missing, and then fix things until everything is linked or you are comfortable ignoring the unlinked edges.

What can we do with unlinked edges? There are several options.

For unlinked `biosphere` edges, we can add these edges to a new database, creating new elementary flows. The impact categories don't know about these flows, so they won't be characterized (unless we add CFs ourselves manually), but they won't be deleted:

`.create_new_biosphere("<new database name>")`

We can also create these new flows in an existing database:

`.add_unlinked_flows_to_biosphere_database("<existing database name>")`

Similarly, for missing products or processes we can add them to the database we are currently importing:

`.add_unlinked_flows_to_biosphere_database()`

Finally, we can also just delete unlinked edges if we know that they won't be important. Because this is a destructive operation we need to pass a flag saying that we are sure it's OK:

`.drop_unlinked(i_am_reckless=True)`

To get some examples of this pattern with real data, look at [the notebooks here whose name starts with `IO`](https://github.com/brightway-lca/brightway2/tree/master/notebooks).

# Real world data: BAFU 2025

Let's try with some real world data.

In [ ]:
bi.install_project("ecoinvent-3.12-biosphere", project_name="BAFU 2025", overwrite_existing=True)

In [ ]:
bd.projects.set_current("BAFU 2025")

In [ ]:
bafu = bi.SingleOutputEcospold1Importer(
    "/etc/data/bafu/",
    "BAFU 2025",
)

Let's check the size of the database:

In [ ]:
bafu.statistics()

We don't always have to do the linking work ourselves. We can use pre-existing matchings compiled automatically or manually. We have a collection of these in the [randonneur_data](https://github.com/brightway-lca/randonneur_data).

Let's look quickly at [randonneur](https://github.com/brightway-lca/randonneur) before diving into the data.

Now that we have a sense of randonneur, let's look at some of the pre-existing matchings:

In [ ]:
import randonneur_data as rd

In [ ]:
registry = rd.Registry()

In [ ]:
list(registry)[:5]

In [ ]:
registry.sample('simapro-ecoinvent-3.10-cutoff')

In [ ]:
registry.sample('ecoinvent-3.11-cutoff-ecoinvent-3.12-cutoff')

In [ ]:
registry.sample('Flowmapper-standard-units-harmonization')

For the BAFU database, we have prepared a mapping of elementary flows to the ecoinvent 3-12 biosphere using [flowmapper](https://github.com/cauldron/flowmapper). This is a local file, not part of `randonneur_data`.

In [ ]:
import randonneur as rn

In [ ]:
bafu.randonneur(
    datapackage=rn.Datapackage.from_json(Path("data/bafu-2025-ecoinvent-3.12-biosphere.json")),
    mapping={
        "source": {"context": "categories"}, 
        "target": {"context": "categories"}
    },
    fields=["name", "categories", "unit"],
)

This transformation add the field "identifier" to the elementary flow edges - we can then use that to create the necessary `input` attribute:

In [ ]:
def use_identifier(data: list) -> list:
    for ds in data:
        for exc in ds["exchanges"]:
            if exc["type"] == "biosphere" and "identifier" in exc:
                exc["input"] = ("ecoinvent-3.12-biosphere", exc["identifier"])
    return data

In [ ]:
bafu.apply_strategy(use_identifier)

We can now apply our generic strategies, which will do some data harmonization, and internal linking in the BAFU database:

In [ ]:
bafu.apply_strategies()

We have some leftovers from the SimaPro mental model of putting everything into the name field. Let's separate out the location, so the name is the name.

In [ ]:
bi.strategies.split_simapro_name_geo_curly_brackets??

In [ ]:
bafu.apply_strategy(bi.strategies.split_simapro_name_geo_curly_brackets)

And do internal matching again:

In [ ]:
bafu.match_database(fields=["name", "unit", "location"])

In [ ]:
bafu.statistics()

We still have some unlinked technosphere edges. Let's look at them:

In [ ]:
for edge in bafu.unlinked:
    if edge['type'] == 'technosphere':
        print(edge)

Let's investigate why these aren't matching by looking at the respective potential matches:

In [ ]:
[
    (x['name'], x['location'], x['unit'])
    for x in bafu.data
    if x['name'] == 'Natural gas, at production'
    and x['location'] == 'DZ'
]

In [ ]:
[
    (x['name'], x['location'], x['unit'])
    for x in bafu.data
    if x['name'] == 'Use, computer, desktop with LCD monitor, active mode'
    and x['location'] == 'CH'
]

In [ ]:
[
    (x['name'], x['location'], x['unit'])
    for x in bafu.data
    if x['name'] == 'Electricity, natural gas, at power plant'
    and x['location'] == 'FI'
]

In [ ]:
[
    (x['name'], x['location'], x['unit'])
    for x in bafu.data
    if x['name'] == 'Transport, freight helicopter, single-engine'
    and x['location'] == 'CH'
]

We could write a simple strategy to fix these unit conversions:

In [ ]:
TO_YEAR = (
    'Transport, freight helicopter, single-engine',
    'Use, computer, desktop with LCD monitor, active mode',
)

def fix_units(data: list) -> list:
    for ds in data:
        for exc in ds['exchanges']:
            if exc.get("input"):
                continue
            elif (
                exc['name'].startswith("Natural gas,") 
                and exc.get("categories", [None])[0] == 'natural gas' 
                and exc["unit"] == 'cubic meter'
            ):
                exc["unit"] = 'normal cubic meter'
            elif exc['name'] in TO_YEAR and exc["unit"] == "hour":
                bi.utils.rescale_exchange(exc, 1/8760)
                exc["unit"] = "year"
            elif (
                exc['name'] == 'Electricity, natural gas, at power plant' 
                and exc['location'] == 'FI' 
                and exc["unit"] == "kilowatt hour"
            ):
                bi.utils.rescale_exchange(exc, 3.6)
                exc["unit"] = "megajoule"
    return data

In [ ]:
bafu.apply_strategy(fix_units)

This fix could also be done using `randonneur`.

Let's retry linking with our updated values:

In [ ]:
bafu.match_database(fields=["name", "unit", "location"])

In [ ]:
bafu.statistics()

We can create a new database for the unlinked elementary flows:

In [ ]:
bafu.create_new_biosphere("BAFU-2025-biosphere")

## Step 3: Loading the data into the database

This, at least, is easy.

In [ ]:
bafu.write_database()

Check to make sure we can do a calculation:

In [ ]:
import bw2calc as bc

In [ ]:
lca = bc.LCA({bd.Database("BAFU 2025").random(): 1}, bd.methods.random())
lca.lci()
lca.lcia()
lca.score